# Проект 1.
# Бизнес-вопрос: Окупилось ли подорожание 1 марта 2026 года

С 1 марта 2026 года тариф «Базовый» подорожал с 199 до 249 ₽, «Расширенный» - с 399 до 449 ₽. Нужно понять, привело ли увеличение стоимости тарифа к увеличению дохода, или отток клиентов съел всю разницу. Отличался ли эффект для двух разных тарифов?

### Гипотезы

**Гипотеза 1.** 

Рост цен на тарифы увеличил отток, база активных пользователей сократилась, а прибыль снизилась.

**Гипотеза 2.**

В процентном соотношении, цена "Базового" увеличилась вдвое больше, чем цена "Расширенного" - +25% и +12,5% соответственно. Можно предположить, что отток клиентов "Базового" был больше - они отказались от подписки или перешли на "Расширенный".


---
## 1. Подключение и параметры

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine   

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC — так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
# Все параметры анализа — в одном месте. Поменяли здесь — пересчиталось всё ниже.

TARIFF = {1: "Базовый", 2: "Расширенный"}          
COLORS = {"Базовый": "#C1272D", "Расширенный": "#9CA3AF"}   

PRICE_CHANGE = "2026-03-01"                       

# Окна сравнения: по три месяца до и после подорожания. Правый конец не включён.
BEFORE = ("2025-12-01", "2026-03-01")             # декабрь, январь, февраль
AFTER = ("2026-03-01", "2026-06-01")              # март, апрель, май

# Длинный ряд для графиков: с сентября 2025 по июль 2026 (месяцы, от первого до последнего).
FIRST_MONTH, LAST_MONTH = "2025-09-01", "2026-07-01"

###  Почему такие окна

Подорожание случилось 01.03.2026, поэтому для анализа беру два временных промежутка по 3 месяца каждый: с 01.12 до 01.03 (конец не включен) и с 01.03 до 01.06. Важно отметить, что периоды не абсолютно равны по продолжительности. Из предудущего ДЗ мы знаем, что в данных за апрель нет двух дней, а сами месяцы разной продолжительности. Поэтому все данные будем сравнивать в расчете на день. 


---
## 2. Аудит данных

Проверим две вещи, чтобы предположения, которые мы сделали при постановке гипотезы, были подтверждены числами.

**Есть ли дни без платежей?** - в ДЗ-2 в апреле не хватало двух дней.

**Действительно ли цена изменилась 1 марта?** - сравниваем суммы, которые пользователи реально платили по двум тарифам


In [ ]:
# Успешные платежи по дням за весь длинный ряд.
days = q("""
    select paid_at::date as day, count(*) as n_payments
    from prime.payments
    where status = 'success'
      and paid_at >= cast(:a as timestamptz)
      and paid_at <  cast(:b as timestamptz) + interval '1 month'
    group by 1
    order by 1
""", a=FIRST_MONTH, b=LAST_MONTH)

days["day"] = pd.to_datetime(days["day"])

# Все календарные дни периода и те из них, которых нет в платежах.
calendar = pd.date_range(FIRST_MONTH, pd.Timestamp(LAST_MONTH) + pd.offsets.MonthEnd(0), freq="D")
missing = calendar.difference(days["day"])
print("дней в календаре:", len(calendar), "| дней с платежами:", len(days))
print("дни без единого платежа:", [d.strftime("%Y-%m-%d") for d in missing])

# Сколько дней с данными в каждом месяце
days["month"] = days["day"].dt.strftime("%Y-%m")
days_with_data = days.groupby("month").size().rename("days_data")
days_with_data

In [ ]:
# Какие суммы платили по каждому тарифу до и после 1 марта.
# Одна строка — тариф × период × сумма платежа; n — сколько таких платежей.
prices = q("""
    select s.tariff_id,
           case when p.paid_at < cast(:cut as timestamptz) then '1. до' else '2. после' end as period,
           p.amount,
           count(*) as n
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success'
      and p.paid_at >= cast(:a as timestamptz) and p.paid_at < cast(:b as timestamptz)
    group by 1, 2, 3
""", cut=PRICE_CHANGE, a=BEFORE[0], b=AFTER[1])

prices["tariff"] = prices["tariff_id"].map(TARIFF)
prices["share_%"] = (prices["n"] / prices.groupby(["tariff", "period"])["n"].transform("sum") * 100).round(2)

# Три самые частые суммы в каждой клетке «тариф × период».
(prices.sort_values("n", ascending=False)
       .groupby(["tariff", "period"]).head(3)
       .sort_values(["tariff", "period", "n"], ascending=[True, True, False])
       [["tariff", "period", "amount", "n", "share_%"]])

### Результат аудита

Видим, что отсутствуют данные по двум датам в апреле - это я уже выяснил, когда делал ДЗ. Кроме того, стоит отметить, что количество дней c данными в двух наших отрезках получилось равным - по 90 дней (если вычесть из второго отрезка 2 пропущенных дня). Однако, поскольку продолжительность самих месяцев все равно разная, нормализация с приведением показателей к одному дню оправдана.

Показатель `share_%` указывает, что все платежи до и после cut однородны, следовательно, цена действительно изменилась 1 марта.

Из второй части аудита видно, что количество платежей по обоим тарифам, и по "Базовому", и по "Расширенному" увеличилось после изменения цен. Следовательно, база не сократилась - это частично опровергает H1. Далее проверим, увеличился ли отток.

---
## 3. Месячная таблица: зерно «месяц × тариф»

#### Почему такое зерно:
Наш бизнес-вопрос сформулирован вокруг разницы результатов по двум тарифам: рост цен был разным для двух тарифов и нам нужно понять, повлияло ли это на поведение наших подписчиков. Нас интересует именно финансовый результат нашего продукта в целом, а не результат по усредненному клиенту - поэтому тариф - оптимальная единица.

Мы знаем, что изменеие цен произошло 1 марта и ведем отчет от этой даты. Платеж происходит ежемесячно. Поэтому месяц - оптимальная единица времени для сравнения базы клиентов и оттока.


Одна строка - один месяц одного тарифа. В строке:

| Колонка | Данные |
|---|---|
| `active` | подписки, активные на протяжении месяца |
| `active_start` | подписки, активные на первое число месяца — показатель оттока |
| `churned` | из `active_start` - сколько закончились в этом месяце |
| `started` | новые подписки, начавшиеся в этом месяце |
| `n_payments`, `revenue` | число успешных платежей и выручка |

Таблицу участников `subscription_members` не трогаем: если присоединить платежи к участникам,
один платёж размножится на всех людей подписки и возникнет двойной счет.

In [ ]:
monthly = q("""
    with months as (
        select m.month_start
        from generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as m(month_start)
    ),
    act as (
        select to_char(m.month_start, 'YYYY-MM') as month,
               s.tariff_id,
               count(*) as active,
               count(*) filter (where s.started_at < m.month_start) as active_start,
               count(*) filter (where s.started_at < m.month_start
                                  and s.ended_at < m.month_start + interval '1 month') as churned,
               count(*) filter (where s.started_at >= m.month_start) as started
        from months m
        join prime.subscriptions s
          on s.started_at < m.month_start + interval '1 month'
         and (s.ended_at is null or s.ended_at >= m.month_start)
        group by 1, 2
    ),
    rev as (
        select to_char(p.paid_at, 'YYYY-MM') as month,
               s.tariff_id,
               count(*) as n_payments,
               sum(p.amount) as revenue
        from prime.payments p
        join prime.subscriptions s on s.subscription_id = p.subscription_id
        where p.status = 'success'
          and p.paid_at >= cast(:first as timestamptz)
          and p.paid_at <  cast(:last as timestamptz) + interval '1 month'
        group by 1, 2
    )
    select a.month, a.tariff_id, a.active, a.active_start, a.churned, a.started,
           coalesce(r.n_payments, 0) as n_payments,
           coalesce(r.revenue, 0) as revenue
    from act a
    left join rev r on r.month = a.month and r.tariff_id = a.tariff_id
    order by 1, 2
""", first=FIRST_MONTH, last=LAST_MONTH)

print(len(monthly), "строк — ожидаем 11 месяцев × 2 тарифа = 22")
monthly.head()

In [ ]:
# Проверим на потерю и двойной счёт денег: выручка в нашей таблице обязана совпасть с суммой успешных платежей, посчитанной напрямую по таблице payments, без соединений.
control = q("""
    select sum(amount) as revenue
    from prime.payments
    where status = 'success'
      and paid_at >= cast(:first as timestamptz)
      and paid_at <  cast(:last as timestamptz) + interval '1 month'
""", first=FIRST_MONTH, last=LAST_MONTH).iloc[0, 0]

diff = monthly["revenue"].sum() - control
print(f"выручка в таблице: {monthly['revenue'].sum():,.2f} ₽".replace(",", " "))
print(f"контроль по payments: {control:,.2f} ₽".replace(",", " "))
assert abs(diff) < 0.01, f"деньги не сходятся на {diff:.2f} ₽"
print("ДЕНЬГИ сходятся: двойного счёта и потерь нет")

# Каждый месяц обязан встретиться ровно по разу на тариф.
assert not monthly.duplicated(["month", "tariff_id"]).any(), "дубли в зерне месяц × тариф"

#### Теперь можем посчитать производные метрики с пересчетом на день ####
`revenue_per_day` - выручка за 1 день (учитываются только дни с данными)

`churn_30d` - отток за 30 дней в процентах - доля тех, кто ушел от общего числа активных пользователей на начало месяца, приведенная к 30 дням

`new_per_day` - количество новых подписок за 1 день

In [ ]:
monthly["tariff"] = monthly["tariff_id"].map(TARIFF)
monthly["days"] = pd.to_datetime(monthly["month"]).dt.days_in_month   
monthly["days_data"] = monthly["month"].map(days_with_data)           

monthly["revenue_per_day"] = monthly["revenue"] / monthly["days_data"]

monthly["churn_30d"] = monthly["churned"] / monthly["active_start"] / monthly["days"] * 30 * 100

monthly["new_per_day"] = monthly["started"] / monthly["days"]

monthly[["month", "tariff", "active", "revenue_per_day", "churn_30d", "new_per_day"]].round(2)

---
## 4. Динамика выручки - график

In [ ]:
def line_by_tariff(col: str, title: str, ylabel: str) -> plt.Axes:
    """Линия по месяцам для каждого тарифа и пунктир в день подорожания."""
    wide = monthly.pivot_table(index="month", columns="tariff", values=col, aggfunc="sum")
    x = range(len(wide))                                  # месяцы — по порядку, 0, 1, 2…

    fig, ax = plt.subplots(figsize=(9, 4.4), layout="constrained")
    for name, color in COLORS.items():
        ax.plot(x, wide[name], marker="o", linewidth=2, color=color, label=name)

    # Пунктир между февралём и мартом.
    cut = list(wide.index).index(PRICE_CHANGE[:7]) - 0.5
    ax.axvline(cut, color="#111827", linestyle="--", linewidth=1)
    ax.text(cut + 0.1, ax.get_ylim()[1], "подорожание 1 марта", va="top", fontsize=9)

    ax.set_xticks(list(x), wide.index, rotation=45)
    ax.set_title(title, loc="left")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("месяц")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#E5E7EB")
    ax.legend(frameon=False)
    return ax


# Делим на 1000, чтобы ось читалась в тысячах рублей.
monthly["revenue_per_day_k"] = monthly["revenue_per_day"] / 1000

TITLE_1 = "График 1. Выручка увеличилась для обоих тарифов"
line_by_tariff("revenue_per_day_k", TITLE_1, "тыс. ₽ в день")
plt.show()

График показывает, что после Мартовского повышения цен выручка по обоим тарифам выросла. Причем рост для тарифа "Базовый" значительно выше (~ 700 ->900 тыс./день против ~860->990 тыс./день) - и это объяснимо при том, что в процентном соотношении цена "Базового" выросла сильнее. Однако, чтобы окончательно отвергнуть или же частично подтвердить H1, нам необходимо понять, за счет чего выросла выручка и посчитать, увеличился ли отток.

## 5. Факторное разложение: за счёт чего изменилась выручка

Сравниваем выручку в пересчете на один день за два наших временных отрезка - до и после повышения цены.
Мультипликативная модель взята из С6:

 > выручка в день = активные подписки × платежей на подписку в день × цена платежа

Сначала складываем данные по тарифам, внутри тарифа раскладываем
на множители (за счёт чего).
Цену раскладываем внутри тарифа, поэтому сдвиг долей тарифов не попадает в «цену»

In [ ]:
def contrib(x0: dict, x1: dict, order: list[str]) -> dict:
    """Цепные подстановки (функция с семинара 6): меняем факторы по одному в порядке order."""
    cur, out = dict(x0), {}
    for f in order:
        before = math.prod(cur.values())     # показатель до замены фактора
        cur[f] = x1[f]                       # фактор переходит на новое значение, остальные стоят
        out[f] = math.prod(cur.values()) - before
    total = math.prod(x1.values()) - math.prod(x0.values())
    assert abs(sum(out.values()) - total) < 0.01, "сумма влияний не равна изменению"
    return out


def window_factors(start: str, end: str) -> pd.DataFrame:
    """Факторы выручки по тарифам за окно [start, end)."""
    w = monthly[(monthly["month"] >= start[:7]) & (monthly["month"] < end[:7])]
    g = w.groupby("tariff").agg(
        active=("active", "mean"),          
        n_payments=("n_payments", "sum"),
        revenue=("revenue", "sum"),
        days_data=("days_data", "sum"),     
    )
    g["per_sub"] = g["n_payments"] / (g["active"] * g["days_data"])   
    g["price"] = g["revenue"] / g["n_payments"]                       
    g["revenue_per_day"] = g["revenue"] / g["days_data"]
    return g


FACTORS = ["active", "per_sub", "price"]
LABELS = {"active": "активные подписки", "per_sub": "платежей на подписку", "price": "цена платежа"}

f0 = window_factors(*BEFORE)
f1 = window_factors(*AFTER)

print("дней с данными: до —", int(f0["days_data"].iloc[0]), "| после —", int(f1["days_data"].iloc[0]))
pd.concat({"до": f0, "после": f1})[["active", "per_sub", "price", "revenue_per_day"]].round(4)

In [ ]:
# Влияние каждого фактора внутри каждого тарифа, руб. в день.
effects = pd.DataFrame({
    t: contrib(f0.loc[t, FACTORS].to_dict(), f1.loc[t, FACTORS].to_dict(), FACTORS)
    for t in f0.index
}).T

# Сумма шести влияний обязана равняться изменению всей выручки в день.
total_delta = f1["revenue_per_day"].sum() - f0["revenue_per_day"].sum()
assert abs(effects.to_numpy().sum() - total_delta) < 0.01, "сумма влияний не равна изменению выручки"

table = effects.rename(columns=LABELS)
table["итого по тарифу"] = table.sum(axis=1)
table.loc["итого"] = table.sum()

print("Влияние факторов, ₽ в день. Порядок подстановок: активные -> частота -> цена.")
print(f"Изменение выручки в день: {total_delta:,.0f} ₽".replace(",", " "))
print(f"То же за 30 дней: {total_delta * 30:,.0f} ₽".replace(",", " "))
table.round().astype(int)

In [ ]:
def waterfall(deltas: pd.Series, title: str) -> plt.Axes:
    """Каскад изменений : по столбику на фактор и «итого» в конце."""
    before = deltas.cumsum() - deltas      # уровень, на котором закончился предыдущий столбик
    colors = ["#C1272D" if v < 0 else "#2A9D8F" for v in deltas]

    fig, ax = plt.subplots(figsize=(10, 4.8), layout="constrained")
    bars = ax.bar(deltas.index, deltas, bottom=before, color=colors, zorder=2)
    total = ax.bar("итого", deltas.sum(), color="#6B7280", zorder=2)
    for group in (bars, total):
        ax.bar_label(group, fmt=lambda v: f"{v:+,.0f}".replace(",", " ").replace("-", "−"), padding=3)
    ax.axhline(0, color="#111827", linewidth=0.8)
    ax.set_title(title, loc="left")
    ax.set_ylabel("тыс. ₽ в день")
    ax.use_sticky_edges = False
    ax.margins(y=0.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#E5E7EB")
    ax.tick_params(axis="x", labelrotation=20)
    return ax


# Шесть столбиков: тариф × фактор, в тысячах рублей в день.
steps = effects.rename(columns=LABELS).stack() / 1000
steps.index = [f"{t}:\n{f}" for t, f in steps.index]

TITLE_2 = "График 2. Цена платежа - главный фактор роста выручки"
ax = waterfall(steps, TITLE_2)
ax.set_xlabel("Дек–фев → мар–май 2026, выручка на день с данными. Порядок подстановок: активные → частота → цена.")
plt.show()

#### Результаты

Ранее мы выяснили, что общая выручка выросла для обоих тарифов, причем для "Базового" рост в процентном соотношении был больше. 

Делаем факторную декомпозицию по трем основным факторам внутри каждого тарифа:
`active` - среднее количество активных подписок внутри каждого тарифа за окно (3 месяца)
`per_sub` - сколько платежей за подписку проходит за день (1/30)
`price` - цена одного платежа

Смотрим влияние каждого из факторов по модели из С6.

#### Что показало факторное разложение:

- После 1 Марта выручка в пересчете на день увеличилась на 409 тыс. рублей: из них ~72% дало увеличение цен, и ~28% - рост базы.
- Самый сильный эффект дало увеличение цены. Если исключить прочие факторы, оно принесло бы 183 тыс. рублей в день для Базового и 112 тыс. рублей для Расширенного.
- Увеличение базы, наоборот, дало больший эффект для "Расширенного" - если бы цена не менялась, только от увеличения базы Расширенный бы прибавил в выручке 64 тыс. рублей, а базовый - 50 тыс. рублей в день. Это легко объясняется изначальной разницей в цене.
- Частота платежей изменилась незначительно. Мы наблюдаем очень малое снижение транзакций по "Расширенному" (около 0.05%), что в целом указывает нам на то, что после повышения цен доля успешных транзакций почти не измелилась.

Возвращаясь к гипотезам, можно опровергнуть предположение о том, что база уменьшилась - эффект базы положительный, и дополнительную выручку от увеличения цены он не "съел". Это еще раз частично опровергает H1. Для подтверждения или опровердения гипотез осталось сравнить отток по двум тарифам.

---
## 6. Вырос ли отток после подорожания?

Отток - доля подписок, которые были активны в первое число, но не возобновились в течение месяца (Показатель приведен к 30 дням)

In [ ]:
TITLE_3 = "График 3. Рост оттока 'Расширенного', снижение оттока 'Базового'"
line_by_tariff("churn_30d", TITLE_3, "% подписок за 30 дней")
plt.show()

In [ ]:
def window_summary(start: str, end: str) -> pd.DataFrame:
    """Отток и приток подписок за окно, по тарифам."""
    w = monthly[(monthly["month"] >= start[:7]) & (monthly["month"] < end[:7])].copy()
    w["sub_days"] = w["active_start"] * w["days"] 
    g = w.groupby("tariff").agg(churned=("churned", "sum"), sub_days=("sub_days", "sum"),
                                started=("started", "sum"), days=("days", "sum"))
    g["churn_30d_%"] = g["churned"] / g["sub_days"] * 30 * 100
    g["new_per_day"] = g["started"] / g["days"]
    return g


s0, s1 = window_summary(*BEFORE), window_summary(*AFTER)

h1 = pd.DataFrame({
    "отток до, %": s0["churn_30d_%"],
    "отток после, %": s1["churn_30d_%"],
    "новых в день до": s0["new_per_day"],
    "новых в день после": s1["new_per_day"],
})
h1["отток, изменение п.п."] = h1["отток после, %"] - h1["отток до, %"]
h1.round(2)

Видим, что у "Базового" отток не только не увеличился - а уменьшился после 1 марта. Это позволяет нам окончательно отвергнуть Гипотезу 1.

Видим, что у "Расширенного" произошло небольшое повышение оттока на 0.05пп., однако, судя по графику, подобные скачки возникали и раньше (см. "Базовый" в ноябре 2025), поэтому однозначно связать это явление с ростом цен нельзя.

Дополнительно, можно отметить, что отток в обоих тарифах увеличивался с января по февраль 2025 года. Можно было бы предположить, что некоторые пользователи отменили подписку заранее, ожидая роста цен. Однако нам не известно, были ли пользователи проинформированы, поэтому предположений строить не будем.